In [2]:
import numpy as np
from ase.io import read, write

datasets = [
    # "Graphene",
    "Diamond",
    "Graphite",
    "Nanotubes",
    "Fullerenes",
    "Liquid",
]

for dataset in datasets:
    inp = f"/home/grethel/dev/quests/examples/gap20/{dataset}.xyz"
    out = f"/home/grethel/dev/quests/examples/gap20_reflect_invert/{dataset}_reflect_invert.xyz"

    frames = read(inp, index=":")  # reads first frame; if your file has many frames, see note below

    transformed_frames = []
    for atoms0 in frames:
        H = atoms0.cell.array  # 3x3 lattice matrix (rows are lattice vectors in ASE)
        Hinv = np.linalg.inv(H)

        # Use DIRECT (fractional) coordinates
        # lattice vector
        s0 = atoms0.get_scaled_positions(wrap=False)  # shape (N,3)
        F0 = atoms0.arrays['force']        # shape (N,3), Cartesian components

        # Choose the center of the operations in fractional coordinates:
        # c = np.array([0.0, 0.0, 0.0])   # about origin
        c = np.array([0.5, 0.5, 0.5])     # about cell center (recommended)

        # 8 sign-flip matrices corresponding to:
        # ( x,  y,  z), ( x,  y, -z), ..., (-x, -y, -z)
        signs = [
            (1, 1, 1),
            (1, 1, -1),
            (1, -1, 1),
            (1, -1, -1),
            (-1, 1, 1),
            (-1, 1, -1),
            (-1, -1, 1),
            (-1, -1, -1),
        ]

        for sx, sy, sz in signs:
            M = np.diag([sx, sy, sz])          # acts in fractional basis
            # Change of basis: fractional <- Cartesian <- fractional
            A = H @ M @ Hinv 
            # Transform fractional positions about center c, then wrap into [0,1)
            # s0 - c makes positions relative to center c, so that the operation treats c as origin
            # (s0 - c) @ M.T applies the operation w.r.t. c as origin and position s0 relative to c
            # c + (s0 - c) @ M.T. shifts origin back to original origin
            s_new = c + (s0 - c) @ M.T
            # The operation might push a position outside of the unit cell [0,1), so we wrap it back by
            # applying modulo 1.0, e.g. (1.07, -0.12, 0.83) -> (0.07, 0.88, 0.83)
            s_new = s_new % 1.0

            # Transform forces (Cartesian vectors)
            F_new = F0 @ A.T

            at = atoms0.copy()
            at.set_scaled_positions(s_new)     # cell unchanged; PBC preserved
            at.set_array("force", F_new)       # keep extxyz "force" property consistent

            # Optional: label which transform this is
            # at.info["config_type"] = f"signflip_{sx}{sy}{sz}"

            transformed_frames.append(at)

    write(out, transformed_frames)

    print(f"Wrote {len(transformed_frames)} frames to {out}")

Wrote 2160 frames to /home/grethel/dev/quests/examples/gap20_reflect_invert/Diamond_reflect_invert.xyz
Wrote 1280 frames to /home/grethel/dev/quests/examples/gap20_reflect_invert/Graphite_reflect_invert.xyz
Wrote 9800 frames to /home/grethel/dev/quests/examples/gap20_reflect_invert/Nanotubes_reflect_invert.xyz
Wrote 21496 frames to /home/grethel/dev/quests/examples/gap20_reflect_invert/Fullerenes_reflect_invert.xyz
Wrote 4104 frames to /home/grethel/dev/quests/examples/gap20_reflect_invert/Liquid_reflect_invert.xyz


In [ ]:
import numpy as np
from ase.io import iread

datasets = [
    "Graphene",
    "Diamond",
    "Graphite",
    "Nanotubes",
    "Fullerenes",
    "Liquid",
]

models = [
    "mace_mp_small",
    "mace_mp_medium",
    "mace_mp_large",
    "mace_off_small",
    "mace_off_medium",
    "mace_off_large",
    "uma-s-1p1",
    "uma-m-1p1",
    # "eqV2_31M_omat_mp_salex",
    # "eqV2_86M_omat_mp_salex",
    # "eqV2_153M_omat_mp_salex",
    # "eqV2_dens_31M_mp",
    # "eqV2_dens_86M_mp",
    # "eqV2_dens_153M_mp",
    # "orb-v3-conservative-inf-omat",
    # "orb-v3-conservative-20-omat",
    # "orb-v3-conservative-inf-mpa",
    # "orb-v3-conservative-20-mpa",
]

for dataset in datasets:
    for model in models:
        print(f"Processing dataset={dataset}, model={model}")
        out_path = f"/home/grethel/dev/quests/embeddings/npz/reflect_invert_invariant/{model}_{dataset}_reflect_invert_invariant.npz"
        # -----------------------------
        # Load frames and embeddings
        # -----------------------------
        frames = list(iread(
            f"/home/grethel/dev/quests/examples/gap20_reflect_invert/{dataset}_reflect_invert.xyz",
            format="extxyz"
        ))

        try:
            X = np.load(
                f"/home/grethel/dev/quests/embeddings/npz/reflect_invert/{model}_{dataset}_reflect_invert.npz",
                allow_pickle=True
            )["embeddings"]
        except:
            print(f"--------Skipping: embeddings for model={model}, dataset={dataset} not found")
            continue

        natoms = np.array([len(f) for f in frames])

        # Find contiguous blocks of constant atom count
        blocks = []
        start = 0
        for i in range(1, len(natoms)):
            if natoms[i] != natoms[i - 1]:
                blocks.append((start, i, natoms[i - 1]))
                start = i
        blocks.append((start, len(natoms), natoms[-1]))

        # -----------------------------
        # Frame → atom offsets
        # -----------------------------
        offsets = np.zeros(len(natoms) + 1, dtype=int)
        offsets[1:] = np.cumsum(natoms)

        # -----------------------------
        # Symmetry averaging (groups of 8)
        # -----------------------------
        avg_embeddings = []

        for b0, b1, n in blocks:
            if (b1 - b0) % 8 != 0:
                raise ValueError(f"Frames {b0}-{b1} not divisible by 8")

            for i in range(b0, b1, 8):
                sym = np.stack([
                    X[offsets[j]:offsets[j + 1]]
                    for j in range(i, i + 8)
                ])
                avg_embeddings.append(sym.mean(axis=0))

        X_invariant = np.concatenate(avg_embeddings, axis=0)

        print("Invariant embedding shape:", X_invariant.shape)
        np.savez(
            out_path,
            embeddings=X_invariant
        )

Processing dataset=Graphene, model=mace_mp_small
Invariant embedding shape: (80800, 128, 16)
Processing dataset=Graphene, model=mace_mp_medium
Invariant embedding shape: (80800, 128, 16)
Processing dataset=Graphene, model=mace_mp_large
Invariant embedding shape: (80800, 256, 16)
Processing dataset=Graphene, model=mace_off_small
Invariant embedding shape: (80800, 96, 16)
Processing dataset=Graphene, model=mace_off_medium
Invariant embedding shape: (80800, 128, 16)
Processing dataset=Graphene, model=mace_off_large
Invariant embedding shape: (80800, 224, 16)
Processing dataset=Graphene, model=uma-s-1p1
Invariant embedding shape: (80800, 9, 128)
Processing dataset=Graphene, model=uma-m-1p1
Invariant embedding shape: (80800, 25, 128)
Processing dataset=Diamond, model=mace_mp_small
Invariant embedding shape: (5400, 128, 16)
Processing dataset=Diamond, model=mace_mp_medium
Invariant embedding shape: (5400, 128, 16)
Processing dataset=Diamond, model=mace_mp_large
Invariant embedding shape: (54